In [2]:
# ... (importaciones y configuración inicial igual)

for x_file in variantes_X:
    base_name = x_file.replace("X_train_", "").replace(".parquet", "")
    try:
        print(f"\n🔍 Procesando: {base_name}")

        # Cargar data
        X_train = pd.read_parquet(os.path.join(BASE_INPUT_PATH, f"X_train_{base_name}.parquet"))
        X_test = pd.read_parquet(os.path.join(BASE_INPUT_PATH, f"X_test_{base_name}.parquet"))
        y_train = pd.read_parquet(os.path.join(BASE_INPUT_PATH, f"y_train_{base_name}.parquet")).squeeze()
        y_test = pd.read_parquet(os.path.join(BASE_INPUT_PATH, f"y_test_{base_name}.parquet")).squeeze()

        # ✅ Remapear las clases a 0-based (0 a 4)
        y_train_mapped = y_train - 1
        y_test_mapped = y_test - 1
        classes = np.unique(y_train_mapped)

        # Calcular scale_pos_weight para clase 0 (originalmente clase 1)
        n_neg = (y_train_mapped != 0).sum()
        n_pos = (y_train_mapped == 0).sum()
        scale_pos_weight = n_neg / n_pos
        print(f"⚖️  scale_pos_weight para clase 1: {scale_pos_weight:.2f}")

        # Entrenamiento
        start = time.time()
        model = xgb.XGBClassifier(objective='multi:softprob', num_class=len(classes),
                                  scale_pos_weight=scale_pos_weight, use_label_encoder=False, random_state=42, n_jobs=-1)

        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        random_search = RandomizedSearchCV(model, param_distributions=param_dist, cv=cv,
                                           scoring='f1_macro', n_iter=10, verbose=1, n_jobs=-1)
        random_search.fit(X_train, y_train_mapped)
        best_model = random_search.best_estimator_
        tiempo = (time.time() - start) / 60

        y_pred = best_model.predict(X_test)
        y_proba = best_model.predict_proba(X_test)
        y_pred_train = best_model.predict(X_train)

        # ✅ Volver a mapear las clases a su valor original (1 a 5)
        y_pred = y_pred + 1
        y_pred_train = y_pred_train + 1
        y_test_original = y_test
        y_train_original = y_train

        # Métricas
        report_test = pd.DataFrame(classification_report(y_test_original, y_pred, output_dict=True)).T
        report_train = pd.DataFrame(classification_report(y_train_original, y_pred_train, output_dict=True)).T
        report_test['set'] = 'test'
        report_train['set'] = 'train'
        full_report = pd.concat([report_train, report_test])
        full_report['tiempo_min'] = tiempo
        full_report.to_csv(os.path.join(OUTPUT_PATH, f"metricas_{base_name}.csv"))

        # PDF Visualizaciones
        with PdfPages(os.path.join(OUTPUT_PATH, f"reporte_{base_name}.pdf")) as pdf:
            ConfusionMatrixDisplay.from_predictions(y_test_original, y_pred, cmap='Blues')
            plt.title("Matriz de Confusión")
            pdf.savefig(); plt.close()

            y_bin = label_binarize(y_test_original, classes=np.arange(1, 6))
            for i, clase in enumerate(range(1, 6)):
                fpr, tpr, _ = roc_curve(y_bin[:, i], y_proba[:, i])
                plt.plot(fpr, tpr, label=f"Clase {clase} (AUC={auc(fpr, tpr):.2f})")
            plt.plot([0, 1], [0, 1], 'k--')
            plt.title("Curvas ROC"); plt.legend(); pdf.savefig(); plt.close()

            for i, clase in enumerate(range(1, 6)):
                precision, recall, _ = precision_recall_curve(y_bin[:, i], y_proba[:, i])
                plt.plot(recall, precision, label=f"Clase {clase}")
            plt.title("Curvas Precision-Recall"); plt.legend(); pdf.savefig(); plt.close()

        # Métricas resumen
        metricas_totales.append({
            "modelo": base_name,
            "accuracy": accuracy_score(y_test_original, y_pred),
            "macro_f1_test": report_test.loc['macro avg', 'f1-score'],
            "macro_f1_train": report_train.loc['macro avg', 'f1-score'],
            "sobreajuste_f1": report_train.loc['macro avg', 'f1-score'] - report_test.loc['macro avg', 'f1-score'],
            "tiempo_min": tiempo
        })

        print(f"✅ {base_name}: F1_macro_test={report_test.loc['macro avg', 'f1-score']:.3f}, Tiempo={tiempo:.2f}min")

    except Exception as e:
        print(f"❌ Error en {base_name}: {e}")

# Guardar resumen global
df_final = pd.DataFrame(metricas_totales)
df_final.to_csv(os.path.join(OUTPUT_PATH, f"resumen_modelos_xgboost.csv"), index=False)
print("\n📊 Resultados guardados correctamente.")



🔍 Procesando: Standard_FE_ALL
⚖️  scale_pos_weight para clase 1: 9.80
Fitting 5 folds for each of 10 candidates, totalling 50 fits
✅ Standard_FE_ALL: F1_macro_test=0.362, Tiempo=194.39min

🔍 Procesando: Standard_FE_ANOVA
⚖️  scale_pos_weight para clase 1: 9.80
Fitting 5 folds for each of 10 candidates, totalling 50 fits
✅ Standard_FE_ANOVA: F1_macro_test=0.366, Tiempo=53.88min

🔍 Procesando: Standard_FE_MI
⚖️  scale_pos_weight para clase 1: 9.80
Fitting 5 folds for each of 10 candidates, totalling 50 fits
✅ Standard_FE_MI: F1_macro_test=0.370, Tiempo=108.63min

🔍 Procesando: Standard_FE_PCA30
⚖️  scale_pos_weight para clase 1: 9.80
Fitting 5 folds for each of 10 candidates, totalling 50 fits
✅ Standard_FE_PCA30: F1_macro_test=0.364, Tiempo=16.21min

🔍 Procesando: Standard_FE_PCAopt
⚖️  scale_pos_weight para clase 1: 9.80
Fitting 5 folds for each of 10 candidates, totalling 50 fits
✅ Standard_FE_PCAopt: F1_macro_test=0.408, Tiempo=320.95min

🔍 Procesando: Standard_FE_VAR
⚖️  scale_pos_